In [15]:
import os
import certifi
from dotenv import load_dotenv

# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [16]:
# LOAD ENV VARIABLES

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

if not GROQ_API_KEY or not TAVILY_API_KEY:
    raise ValueError("Missing API keys in .env")


In [17]:

search_tool = TavilySearchResults(max_results=2) #predfine tool

tools = [search_tool]

print("Tools:", [tool.name for tool in tools])


Tools: ['tavily_search_results_json']


In [18]:
@tool #custome tool

def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )


In [19]:
result = search_tool.invoke("Give me the latest news on Agentic AI")
result


[{'url': 'https://www.taskade.com/blog/agentic-ai',
  'content': 'Key Milestones:\n\n Q1 2024: Gartner inquiry volume on multi-agent systems surged 1,445%\n Q2 2024: Major enterprises began deploying agents for customer service\n Q4 2024: CrewAI secured $18M funding, 60% of Fortune 500 adopted\n Q1 2025: First regulatory guidance on AI agent safety (NIST)\n Q3 2025: Agentic AI market reached $7.5B\n January 2026: 40% of enterprise apps integrated AI agents (Gartner)\n\nProduction Use Cases (2026):'},
 {'url': 'https://www.forbes.com/topics/agentic-ai',
  'content': "# Agentic AI\n\nLatest Agentic AI news freshly curated for you as of 8 a.m. ET Nov. 11. Learn more about innovative developments, discover emerging trends and read expert perspectives.\n\n## Today's Highlights\n\n A boom in agentic security operations is underway, fueled by billions in VC funding to create automated SOCs that can counter sophisticated AI-powered cyberthreats.\n Google Cloud unveiled its Gemini for Healthcar

In [20]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

print(llm.invoke("Hello!").content)

Hello! How can I assist you today?


In [21]:
from langchain.agents import create_openai_tools_agent
from langchain.agents import AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

tools = [search_tool,get_weather_data]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful research assistant. Use web search when needed."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent = create_openai_tools_agent(
    llm, tools, prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [24]:
response = agent_executor.invoke({
    "input": (
        "Find the capital of India."
        "and then find its current weather."
    )
})

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `get_weather_data` with `{'city': 'New Delhi'}`


Could not fetch weather data for New Delhi
Invoking: `get_weather_data` with `{'city': 'New Delhi'}`


Could not fetch weather data for New Delhi
Invoking: `tavily_search_results_json` with `{'query': 'current weather New Delhi'}`


[{'url': 'https://www.ndtv.com/weather/delhi-weather-forecast-today', 'content': '08:00\n\n23°\n\n09:00\n\n25°\n\n10:00\n\n27°\n\n11:00\n\n28°\n\n12:00\n\n28°\n\n13:00\n\n29°\n\n14:00\n\n29°\n\n15:00\n\n29°\n\n16:00\n\n27°\n\n17:00\n\n25°\n\n18:00\n\n25°\n\n19:00\n\n25°\n\n20:00\n\n25°\n\n21:00\n\n24°\n\n22:00\n\n23°\n\n23:00\n\n23°\n\n00:00\n\nSunrise\n\n06:41\n\n10 Hr and 48 Min\n\nSunset\n\n17:29\n\n## 7-Day Delhi Weather Forecast\n\nToday\n\n12/11\n\n28°\n\n17°\n\nFriday\n\n13/11\n\n29°\n\n17°\n\nSaturday\n\n14/11\n\n29°\n\n18°\n\nSunday\n\n15/11\n\n28°\n\n17°\n\nMonday\n\n16/11\n\n27°\n\n15°\n\nTuesday\n\n17/11\n\n26°\n\n14°\n\n### Current Conditions\n\

In [ ]:
# tools = [search_tool, get_weather_data]
# creat Agent
# from langchain.agents import create_react_agent
# from langchain import hub

# prompt = hub.pull("hwchase17/react")

# agent = create_react_agent(
#     llm=llm,
#     tools=tools,
#     prompt=prompt
# )


In [13]:

# from langchain.agents import AgentExecutor

# agent_executor = AgentExecutor(
#     agent=agent,
#     tools=tools,
#     verbose=True,
#     handle_parsing_errors=True
# )


In [14]:

# response = agent_executor.invoke({
#     "input": "Search the web for the latest AI news."
# })

# print(response["output"])
